# R1b — 문장 단위 정답으로 고친 발생 시점 인식 학습

## 왜 하나

R1(10/13)은 재현율 0.541로 올랐지만 지연이 27토큰이었고, 원인은 **라벨 설계 결함**이었다.
정답을 응답 단위로 정해(응답에서 환각이 시작된 뒤의 모든 지점 = 양성), 환각 문장 **뒤의 멀쩡한 문장들**까지 양성으로 학습했다.
판정기 입력은 지금 문장의 앞부분뿐이라, 판정기는 "환각이 시작되는 지점"이 아니라 "수상한 응답을 뒤에서 늦게" 잡는 법을 배웠다
(완성 문장 AUROC 0.857 → 0.78, Data2txt만 상승). 다만 같은 결함 라벨 위에서도 다중 사례 목표(V1)가 이진 분류(V0)보다 +0.061 나았다.

## 무엇을 바꾸나

1. **정답을 문장 단위로** (E2와 같은 정의): 각 문장 안에서 환각 구간과 처음 겹치는 토큰이 그 문장의 시작점.
   - 환각이 있는 문장: 시작 전 = 음성, 시작 후 10토큰 = 창, 그 뒤 = 이후
   - 환각이 없는 문장(멀쩡한 응답이든, 환각 문장의 앞이든 뒤든) = 음성
2. **다중 사례 단위를 문장마다**: V1b는 환각 문장마다 "창 안에서 적어도 하나"를 학습한다(R1은 응답마다 하나).
3. **체크포인트 선택에 지연 조건**: cal-sub에서 4토큰 간격 지연 중앙값 ≤ 9인 체크포인트를 우선, 그중 재현율 최대(없으면 재현율 최대).
4. 변형은 **V0b(지점별 이진 분류, 대조군)와 V1b(창 안 적어도 하나)** 두 가지만, 시드 42·7·2026 모두 학습(6회).
   판정 방식은 학습이 끝난 뒤 cal-L에서 V1b 시드 평균으로 고른다.

## 사전 정의 기준 (결과 보기 전, SSOT §6에 고정 — R1과 같음)

- **R1b-0 정답 점검(설계 결함 재발 방지)**: V0b의 dev-L 완성 문장 AUROC 시드 평균 ≥ 0.84 (E2·B 수준 0.86 근처). 미달이면 라벨·학습에 다시 문제가 있는 것으로 보고 이후 수치를 해석하지 않는다.
- **선택 규칙**: V1b × {문장 끝, 4, 2, 2+EWMA} 중 cal-L 시드 평균 지연 ≤ 9토큰에서 cal-L 시드 평균 재현율 최대. 없으면 지연 조건 없이 재현율 최대(이 경우 주 기준은 지연에서 미달 가능).
- **R1b 주 기준**: dev-L ARL₀=100에서 시드 평균 재현율 ≥ 0.531, 시드 평균 지연 중앙값 ≤ 9토큰, 시드 3개 중 2개 이상에서 B 문장 끝 대비 쌍체 부트스트랩 하한 > 0.
- **목표 효과**: 같은 판정 방식에서 V1b 시드 평균 − V0b 시드 평균 ≥ +0.03.
- **보고(판정 없음)**: 응답 AUROC, 서비스 10% 재현율·지연, 완성 문장 AUROC, 과제별, R1(결함 라벨) 모델과의 비교.

> 런타임: **GPU 필수, A100 또는 L4 권장**. 학습 6회(각 A100 약 10~15분). 모델은 드라이브 `selfproject/r1b/`에 저장되고 끝난 것은 건너뛴다.
> 드라이브에 `student_vark`, R0 결과(`r0/r0_items.pkl`, `r0/r0_scores.npz`)가 있어야 한다. R1 점수(`r1/*/scores.npy`)가 있으면 비교용으로 함께 불러온다.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/selfproject' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
R0_DIR = os.path.join(OUT_DIR, 'r0')
R1_DIR = os.path.join(OUT_DIR, 'r1b')
R1_OLD = os.path.join(OUT_DIR, 'r1')
os.makedirs(R1_DIR, exist_ok=True)
assert os.path.isdir(STUDENT_DIR), f'{STUDENT_DIR} 없음'
print('device', DEVICE)

## 2. 데이터 — gold-large 학습 응답 (E1·E2 코드 그대로)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

## 3. 학습 판정 지점 — 정답은 문장 단위 (E2와 같은 정의)

- `neg`: 환각이 없는 문장의 지점 (멀쩡한 응답, 환각 문장의 앞이든 뒤든)
- `pre`: 환각이 있는 문장에서 시작 전 / `win`: 시작 후 10토큰 안 / `post`: 그 이후
- 다중 사례 단위(V1b): 환각이 있는 **문장마다** 창 하나

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts


WIN_TOK = 10
train_items = to_items(train)
by_tr = {str(r['id']): r for r in train}
TR, P_KEY, P_TXT, P_CAT = [], [], [], []
for x in train_items:
    text = x['text']
    spans = [(l['start'], l['end']) for l in (by_tr[x['id']].get('labels') or [])]
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a for a, b in zip(x['ts'], x['te'])]
    def s_onset(sn):                      # E2와 같다: 이 문장 안에서 환각 구간과 처음 겹치는 토큰
        for t in range(sn['t0'], sn['t1'] + 1):
            if any(min(x['te'][t], b) > max(cs[t], a) for a, b in spans): return t
        return None
    sid = {sn['t0']: i for i, sn in enumerate(x['sents'])}
    son = [s_onset(sn) for sn in x['sents']]
    groups = {}
    for t, txt in points(x, 2):
        if not txt: continue
        si = sid[x['sent_of'][t]]; o = son[si]
        if o is None: cat = 'neg'
        elif t < o: cat = 'pre'
        elif t <= o + WIN_TOK: cat = 'win'
        else: cat = 'post'
        g = len(P_TXT); P_KEY.append(x['ref_key']); P_TXT.append(txt); P_CAT.append(cat)
        if cat in ('win', 'post'): groups.setdefault(si, []).append(g)
    gl = []
    for si, lst in groups.items():
        win = [g for g in lst if P_CAT[g] == 'win']
        if not win:                       # 창 안에 지점이 없으면 시작 이후 첫 지점을 창으로
            P_CAT[lst[0]] = 'win'; win = [lst[0]]
        gl.append(win)
    TR.append({'groups': gl})
P_CAT = np.array(P_CAT)
print(f'학습 응답 {len(TR)} | 판정 지점 {len(P_TXT):,} | 환각 문장(다중 사례 단위) {sum(len(r["groups"]) for r in TR):,} |',
      pd.Series(P_CAT).value_counts().to_dict())
assert sum(len(r['groups']) for r in TR) > 0

## 4. 근거 검색 (E2와 같은 설정) — 지점마다 top-8 조각 번호만 저장

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out


# R0 평가 세트(cal-L, dev-L)와 기존 판정기 점수
import pickle
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
Z0 = dict(np.load(os.path.join(R0_DIR, 'r0_scores.npz')))
CAL, DEV = R0P['items']['calL'], R0P['items']['devL']
Q_KEYS, Q_TEXTS = R0P['q_keys'], R0P['q_texts']
for k in set(Q_KEYS):
    REFS.setdefault(k, build_reference(sources[k]))

@torch.no_grad()
def topk_idx(keys, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            out.append(sorted(torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()))
    return out
ctx_of = lambda key, idx: ' ... '.join(get_chunks(key)[0][j] for j in idx)

t0 = time.perf_counter()
P_IDX = topk_idx(P_KEY, P_TXT)
print(f'학습 지점 근거 검색 {time.perf_counter()-t0:.0f}초')

# 평가 질의: cal-L·dev-L 지점만 (R0와 같은 질의 번호)
EVAL_Q = sorted({q for x in CAL + DEV for _, q in x['pts2']})
t0 = time.perf_counter()
EQ_IDX = dict(zip(EVAL_Q, topk_idx([Q_KEYS[q] for q in EVAL_Q], [Q_TEXTS[q] for q in EVAL_Q])))
print(f'평가 질의 {len(EVAL_Q):,}개 근거 검색 {time.perf_counter()-t0:.0f}초')

# 체크포인트 선택용 cal-sub (cal-L에서 300응답 고정)
CALSUB = random.Random(1).sample(CAL, min(300, len(CAL)))

## 5. 평가 함수 (E1 코드 그대로) + 판정 방식 `2토큰+EWMA`

점수기 이름은 `모델|판정 방식`. 판정 방식: `end`(문장 끝), `4`, `2`, `ew2`(2토큰 간격 점수를 응답 전체에 걸쳐 λ=0.5로 지수 평활).

In [ ]:
SC = dict(Z0)

def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    if k == 'ew2':
        out, s = [], None
        for t, q in x['pts2']:
            v = float(S_[q]); s = v if s is None else 0.5 * v + 0.5 * s
            out.append((t, s))
        return out
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

In [ ]:
def resp_auc(items, m):
    y = [int(x['onset'] is not None) for x in items]
    s = [max([SC[m][q] for _, q in x['pts2']] or [-1e9]) for x in items]
    return roc_auc_score(y, s)

def sent_auc(items, m):
    ys, ss = [], []
    for x in items:
        d = dict(x['pts2'])
        for sn in x['sents']:
            if sn['t1'] in d: ys.append(sn['y']); ss.append(SC[m][d[sn['t1']]])
    return roc_auc_score(ys, ss)

def per_resp(items, sc, thr):
    d, l = [], []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, sc) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        d.append(float(ok)); l.append(first - x['onset'] if ok else np.nan)
    return np.array(d), np.array(l)

def boot(a, b, n=1000, seed=0):
    da, la = per_resp(DEV, a, calibrate(CAL, a, 100)); db, lb = per_resp(DEV, b, calibrate(CAL, b, 100))
    rng = np.random.default_rng(seed); N = len(da); rd = []
    for _ in range(n):
        i = rng.integers(0, N, N); rd.append(da[i].mean() - db[i].mean())
    lo, hi = np.percentile(rd, [2.5, 97.5])
    return {'A': a, 'B': b, 'recall_A': da.mean(), 'recall_B': db.mean(), 'diff': da.mean() - db.mean(), 'lo': lo, 'hi': hi}

# 기준선 확인 (R0 dev-L과 같아야 함)
for sc, ref in [('B|end', 0.5007), ('B|4', 0.4809), ('E2|4', 0.4506)]:
    got = evaluate(DEV, sc, calibrate(CAL, sc, 100))['recall']
    print(f'{sc:6} R0 {ref:.4f} | 지금 {got:.4f} | {"일치" if abs(got-ref) <= 0.002 else "불일치 — 확인 필요"}')

## 6. 학습 — 변형 4가지 공통 루프

한 배치 = 판정 지점 약 64개. 단일 지점 항(이진 분류)과 응답 단위 항(창 안 하나, 순위)을 섞는다.
V2·V3의 순위 항은 매 단계 그 응답의 시작 전 지점을 학습 중인 모델로 먼저 채점해(기울기 없이) 가장 높은 4개를 쓴다.

In [ ]:
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F

BATCH_SEQ, LR, POST_W, GROUP_W, MARGIN, MINE_M, N_EVALS = 64, 2e-5, 0.2, 5.0, 1.0, 4, 4
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16

def encode(tok, idxs):
    ctx = [ctx_of(P_KEY[g], P_IDX[g]) for g in idxs]
    return tok(ctx, [P_TXT[g] for g in idxs], truncation='only_first', max_length=MAX_LEN,
               padding=True, return_tensors='pt').to(DEVICE)

def make_units(variant, rng):
    units = []
    for g in range(len(P_TXT)):
        c = P_CAT[g]
        if variant == 'V0b':
            units.append(('s', g, 1.0 if c in ('win', 'post') else 0.0, 1.0))
        elif c == 'post':
            units.append(('s', g, 1.0, POST_W))
        elif c in ('neg', 'pre'):
            units.append(('s', g, 0.0, 1.0))
    if variant == 'V1b':
        for r in TR:
            for win in r['groups']:
                units.append(('g', win))
    rng.shuffle(units)
    return units

@torch.no_grad()
def score_ids(tok, model, idxs, bs=256):
    model.eval(); out = []
    for i in range(0, len(idxs), bs):
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out.append(model(**encode(tok, idxs[i:i + bs])).logits.squeeze(-1).float().cpu())
    model.train()
    return torch.cat(out).numpy() if out else np.zeros(0)

@torch.no_grad()
def score_queries(tok, model, qs, bs=256):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q_TEXTS[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        ctx = [ctx_of(Q_KEYS[q], EQ_IDX[q]) for q in b]
        enc = tok(ctx, [Q_TEXTS[q] for q in b], truncation='only_first', max_length=MAX_LEN,
                  padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    model.train()
    return out

CALSUB_Q = sorted({q for x in CALSUB for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_queries(tok, model, CALSUB_Q)
    SC['_cur'] = np.zeros(len(Q_KEYS), dtype=np.float32); SC['_cur'][CALSUB_Q] = v
    r4 = evaluate(CALSUB, '_cur|4', calibrate(CALSUB, '_cur|4', 100))
    r2 = evaluate(CALSUB, '_cur|2', calibrate(CALSUB, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

def group_loss(variant, win, tok, model):
    return win, []

def train_variant(variant, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(STUDENT_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(STUDENT_DIR).to(DEVICE)
    units = make_units(variant, random.Random(seed))
    n_seq = sum(1 if u[0] == 's' else 6 for u in units)
    n_steps = max(1, n_seq // BATCH_SEQ)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda' and not BF16))
    best, d0 = calsub_metric(tok, model)
    best_key, best_state, hist = (d0 <= 9, best), copy.deepcopy(model.state_dict()), [(0, best, d0)]
    print(f'  [{variant} s{seed}] 시작 cal-sub 재현율 {best:.3f}, 지연 {d0} | 예상 {n_steps}단계')
    eval_at = {int(n_steps * (i + 1) / N_EVALS) for i in range(N_EVALS)}
    model.train(); step, ui, t0 = 0, 0, time.perf_counter()
    while ui < len(units):
        singles, groups, n = [], [], 0
        while ui < len(units) and n < BATCH_SEQ:
            u = units[ui]; ui += 1
            if u[0] == 's':
                singles.append(u); n += 1
            else:
                win, hard = group_loss(variant, u[1], tok, model)
                if win:
                    groups.append((win, hard)); n += len(win) + len(hard)
        ids = [u[1] for u in singles] + [g for w, h in groups for g in w + h]
        if not ids: continue
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            z = model(**encode(tok, ids)).logits.squeeze(-1).float()
        ns = len(singles)
        y = torch.tensor([u[2] for u in singles], device=DEVICE); w = torch.tensor([u[3] for u in singles], device=DEVICE)
        loss_num = (F.binary_cross_entropy_with_logits(z[:ns], y, reduction='none') * w).sum() if ns else z.sum() * 0
        den = w.sum() if ns else torch.tensor(0.0, device=DEVICE)
        pos = ns
        for win, hard in groups:
            zw = z[pos:pos + len(win)]; zh = z[pos + len(win):pos + len(win) + len(hard)]; pos += len(win) + len(hard)
            s_no = F.logsigmoid(-zw).sum().clamp(max=-1e-6)              # 창 안에서 한 번도 안 울릴 로그확률
            l = -torch.log(-torch.expm1(s_no))                            # 적어도 하나 울릴 확률의 음의 로그
            if len(hard):
                l = l + F.softplus(torch.logsumexp(zh, 0) - torch.logsumexp(zw, 0) + MARGIN)
            loss_num = loss_num + GROUP_W * l; den = den + GROUP_W
        loss = loss_num / den.clamp(min=1e-6)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sch.step()
        step += 1
        if step in eval_at or ui >= len(units):
            va, dv = calsub_metric(tok, model); hist.append((step, va, dv))
            if (dv <= 9, va) > best_key: best_key, best, best_state = (dv <= 9, va), va, copy.deepcopy(model.state_dict())
            print(f'  [{variant} s{seed}] 단계 {step} | 손실 {loss.item():.3f} | cal-sub 재현율 {va:.3f}, 지연 {dv} | {(time.perf_counter()-t0)/60:.1f}분')
    model.load_state_dict(best_state)
    return tok, model, best, hist

import shutil
def run(variant, seed):
    out = os.path.join(R1_DIR, f'{variant}_s{seed}')
    sp = os.path.join(out, 'scores.npy')
    if not os.path.exists(sp):
        has_w = any(os.path.exists(os.path.join(out, f)) for f in ['model.safetensors', 'pytorch_model.bin'])
        if os.path.exists(os.path.join(out, 'config.json')) and not has_w:
            print(f'  [{variant} s{seed}] 저장이 끊긴 폴더(가중치 없음) → 지우고 다시 학습'); shutil.rmtree(out)
        if has_w:
            tok = AutoTokenizer.from_pretrained(out); model = AutoModelForSequenceClassification.from_pretrained(out).to(DEVICE)
        else:
            tok, model, best, hist = train_variant(variant, seed)
            os.makedirs(out, exist_ok=True); tok.save_pretrained(out); model.save_pretrained(out)
            pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(out, 'hist.csv'), index=False)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        print(f'  [{variant} s{seed}] 평가 질의 채점 {time.perf_counter()-t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'{variant}_s{seed}'] = np.load(sp)
    print(f'[{variant} s{seed}] 준비 완료')

## 7. 학습 — V0b·V1b × 시드 42·7·2026 (끝난 것은 건너뜀)

In [ ]:
SEEDS = [42, 7, 2026]
for seed in SEEDS:
    for v in ['V1b', 'V0b']:
        run(v, seed)
# 비교용: R1(결함 라벨) 점수가 드라이브에 있으면 함께 불러온다
for v in ['V0', 'V1']:
    for s in SEEDS:
        p = os.path.join(R1_OLD, f'{v}_s{s}', 'scores.npy')
        if os.path.exists(p): SC[f'R1{v}_s{s}'] = np.load(p)
print('R1 비교 점수:', [k for k in SC if k.startswith('R1')])

## 8. R1b-0 정답 점검과 판정 방식 선택 (cal-L만 사용)

In [ ]:
KINDS = ['end', '4', '2', 'ew2']
v0_sent = float(np.mean([sent_auc(DEV, f'V0b_s{s}') for s in SEEDS]))
R1B0 = v0_sent >= 0.84
print(f"R1b-0 정답 점검: V0b 완성 문장 AUROC 시드 평균 {v0_sent:.3f} (≥ 0.84) → {'충족' if R1B0 else '미달 — 이후 수치 해석 보류'}")

sel = []
for v in ['V0b', 'V1b']:
    for k in KINDS:
        rs = [evaluate(CAL, f'{v}_s{s}|{k}', calibrate(CAL, f'{v}_s{s}|{k}', 100)) for s in SEEDS]
        sel.append({'variant': v, 'kind': k, 'cal_recall': np.mean([r['recall'] for r in rs]),
                    'cal_delay': np.mean([r['delay_tok_median'] for r in rs]),
                    'cal_pre_onset': np.mean([r['pre_onset_alarm_rate'] for r in rs])})
sel_df = pd.DataFrame(sel)
print(sel_df.round(3).to_string(index=False))
elig = sel_df[(sel_df.variant == 'V1b') & (sel_df.cal_delay <= 9)].sort_values('cal_recall', ascending=False)
SEL_FALLBACK = len(elig) == 0
if SEL_FALLBACK:
    print('⚠️ cal-L 지연 ≤ 9토큰인 V1b 판정 방식 없음 → 지연 조건 없이 재현율 최대 (주 기준은 지연에서 미달 가능)')
    elig = sel_df[sel_df.variant == 'V1b'].sort_values('cal_recall', ascending=False)
CHOSEN_V, CHOSEN_K = 'V1b', elig.iloc[0].kind
print(f'\n선택: {CHOSEN_V}, 판정 방식 {CHOSEN_K}')

## 10. dev-L 평가와 사전 기준 판정

In [ ]:
rows = []
models = ['B', 'E2'] + [m for m in Z0 if m.startswith('S')] + [f'{v}_s{s}' for v in ['V0b', 'V1b', 'R1V0', 'R1V1'] for s in SEEDS if f'{v}_s{s}' in SC]
for m in models:
    ra, sa = resp_auc(DEV, m), sent_auc(DEV, m)
    for k in KINDS:
        sc = f'{m}|{k}'
        r = evaluate(DEV, sc, calibrate(CAL, sc, 100))
        r10 = evaluate(DEV, sc, calibrate_flag(CAL, sc, 0.10))
        rows.append({'model': m, 'kind': k, 'recall': r['recall'], 'delay_med': r['delay_tok_median'],
                     'delay_chars': r['delay_chars_median'], 'pre_onset': r['pre_onset_alarm_rate'],
                     'realized_ARL0': r['realized_ARL0_dev'], 'svc_recall': r10['recall'], 'svc_delay': r10['delay_tok_median'],
                     'svc_flag': r10['clean_resp_flagged'], 'resp_auc': ra, 'sent_auc': sa})
res = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print(res[res.kind.isin(['end', '4'])].round(3).to_string(index=False))

def seed_mean(v, k, col='recall'):
    return float(np.mean([res[(res.model == f'{v}_s{s}') & (res.kind == k)][col].iloc[0] for s in SEEDS if f'{v}_s{s}' in SC]))

rec, dl = seed_mean(CHOSEN_V, CHOSEN_K), seed_mean(CHOSEN_V, CHOSEN_K, 'delay_med')
bts = [boot(f'{CHOSEN_V}_s{s}|{CHOSEN_K}', 'B|end') for s in SEEDS]
n_sig = sum(b['lo'] > 0 for b in bts)
c1, c2, c3 = rec >= 0.531, dl <= 9, n_sig >= 2
R1 = c1 and c2 and c3 and R1B0
gain = rec - seed_mean('V0b', CHOSEN_K)
R1_OBJ = gain >= 0.03
print(f"\n=== R1b 사전 기준 ({CHOSEN_V}, {CHOSEN_K}, 시드 {SEEDS}) ===")
print(f"시드 평균 재현율 {rec:.3f} ≥ 0.531: {'충족' if c1 else '미달'}")
print(f"시드 평균 지연 {dl:.1f} ≤ 9: {'충족' if c2 else '미달'}")
for b in bts:
    print(f"   {b['A']}: B 문장 끝 대비 {b['diff']:+.3f} [{b['lo']:+.3f}, {b['hi']:+.3f}]")
print(f"B 문장 끝 대비 유의한 시드 {n_sig}/3 (≥ 2): {'충족' if c3 else '미달'}")
print('R1b 주 기준 (R1b-0 포함):', '충족' if R1 else '미달')
print(f"목표 효과: V0b 대비 시드 평균 {gain:+.3f} (≥ +0.03): {'충족' if R1_OBJ else '미달'}")

agg = []
for v in ['V0b', CHOSEN_V] + [w for w in ['R1V0', 'R1V1'] if f'{w}_s42' in SC]:
    for k in KINDS:
        agg.append({'variant': v, 'kind': k, **{c: seed_mean(v, k, c) for c in
                    ['recall', 'delay_med', 'pre_onset', 'svc_recall', 'svc_delay', 'resp_auc', 'sent_auc']}})
for m, k in [('B', 'end'), ('B', '4'), ('E2', '4')]:
    r = res[(res.model == m) & (res.kind == k)].iloc[0]
    agg.append({'variant': m, 'kind': k, **{c: r[c] for c in ['recall', 'delay_med', 'pre_onset', 'svc_recall', 'svc_delay', 'resp_auc', 'sent_auc']}})
agg_df = pd.DataFrame(agg)
print('\n시드 평균 비교 (dev-L)'); print(agg_df.round(3).to_string(index=False))

## 11. 과제별 (선택된 변형 시드 평균 vs B 문장 끝, 보고)

In [ ]:
tr = []
for tt in TASKS:
    its = [x for x in DEV if x['task_type'] == tt]
    row = {'task': tt, 'n_hall': sum(x['onset'] is not None for x in its)}
    for sc in ['B|end', 'B|4'] + [f'{CHOSEN_V}_s{s}|{CHOSEN_K}' for s in SEEDS]:
        row[sc] = evaluate(its, sc, calibrate(CAL, sc, 100))['recall']
    tr.append(row)
task_df = pd.DataFrame(tr)
print(task_df.round(3).to_string(index=False))

## 12. 저장

In [ ]:
import shutil
summ = {'R1b_0_v0b_sent_auc': v0_sent, 'R1b_0': bool(R1B0), 'chosen_variant': CHOSEN_V, 'chosen_kind': CHOSEN_K, 'selection_fallback': bool(SEL_FALLBACK), 'seed_mean_recall': rec, 'seed_mean_delay': dl,
        'n_sig_vs_Bend': int(n_sig), 'R1': bool(R1), 'gain_vs_V0': gain, 'R1_objective': bool(R1_OBJ),
        'boot': bts}
json.dump(summ, open('r1b_summary.json', 'w'), default=float, ensure_ascii=False, indent=1)
hists = []
for v in ['V0b', 'V1b']:
    for s in SEEDS:
        p = os.path.join(R1_DIR, f'{v}_s{s}', 'hist.csv')
        if os.path.exists(p):
            h = pd.read_csv(p); h['run'] = f'{v}_s{s}'; hists.append(h)
out = {'r1b_devL.csv': res, 'r1b_select_cal.csv': sel_df, 'r1b_seedmean.csv': agg_df, 'r1b_task.csv': task_df,
       'r1b_hist.csv': pd.concat(hists) if hists else pd.DataFrame()}
for f, d in out.items():
    d.to_csv(f, index=False); d.to_csv(os.path.join(R1_DIR, f), index=False)
shutil.copy('r1b_summary.json', os.path.join(R1_DIR, 'r1b_summary.json'))
os.system('zip -q r1b_results.zip ' + ' '.join(list(out) + ['r1b_summary.json']))
print('저장 완료: 드라이브 selfproject/r1b/, 업로드용 r1b_results.zip')
try:
    from google.colab import files; files.download('r1b_results.zip')
except Exception:
    pass